<a href="https://colab.research.google.com/github/lim0119/-2025-3-2-PJ/blob/main/%EB%8B%A8%EC%9C%84_%ED%85%8C%EC%8A%A4%ED%8A%B8(%EC%88%98%EC%A0%95%EB%B3%B8).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pytest
import numpy as np
import time
from typing import List, Dict, Any
from sklearn.model_selection import train_test_split
# train.py에서 실제 구현된 핵심 함수들을 가져옴
# from train import z, crop_or_pad_to, unet3d, cnn3d

# --------------------------------------------------------------------------------
# 임시 함수 (목업): 체크리스트의 기능을 단위 테스트하기 위한 가상 함수
# --------------------------------------------------------------------------------

# 5.1 D_01 검증
def upload_mri_request(file_path: str, patient_info: Dict) -> bool: return file_path.endswith('.dcm')

# 5.3 SEG_03 검증 보조
def segment_image(data: np.ndarray) -> np.ndarray: return np.zeros_like(data)
def calculate_volume_and_split(seg_map: np.ndarray) -> Dict: return {'left_vol': 100, 'right_vol': 100, 'asymmetry': 0.0}

# 5.4 NFE_02 검증
def extract_asymmetry_index(lv: float, rv: float) -> float:
    return abs(lv - rv) / ((lv + rv) / 2) if (lv + rv) > 0 else 0.0
def check_feature_transfer(feature_dict: Dict) -> bool: return 'volume' in feature_dict

# 5.4 NFE_01 검증 보조
def calculate_icv(mri_data: np.ndarray) -> float: return 1500000.0

# 5.7 RVD_03 검증
def save_final_result(patient_info: Dict, result: Dict) -> bool: return patient_info is not None and result is not None

# 5.6 PRE_03 성능 검증 (화면 영상의 "약 3분" 안내에 맞춰 180.0초)
def segment_hippocampus_performance(data_array: np.ndarray, processing_time_sec: float = 180.0) -> np.ndarray:
    time.sleep(processing_time_sec); return np.zeros_like(data_array)

# 5.5 VIS_01 성능 검증
def generate_3d_viewer_performance(seg_map: np.ndarray, processing_time_sec: float = 4.0) -> bool:
    time.sleep(processing_time_sec); return True

# 5.7 RVD_01 검증 보조
def format_prediction_for_ui(prob_dict: Dict) -> str: return f"AD: {prob_dict.get('AD', 0.0)*100:.1f}%"

# [핵심 Mock 함수] train.py에서 사용되는 함수 Mock 정의
def z(arr: np.ndarray) -> np.ndarray: return (arr - np.mean(arr)) / np.std(arr) if np.std(arr) != 0 else arr
def crop_or_pad_to(target_shape: tuple, arr: np.ndarray) -> np.ndarray:
    # 테스트를 위해 shape만 맞추도록 Mock (실제 중앙 크롭/패딩 로직 필요)
    return np.zeros(target_shape, dtype=arr.dtype)

# 5.2 NOR_03 검증: 불필요한 구조(배경) 제거 Mock
def remove_skull_and_background(data: np.ndarray, threshold: float = 0.1) -> np.ndarray:
    clean_data = np.copy(data)
    clean_data[data < threshold] = 0
    return clean_data

# 5.3 SEG_01 검증: 임곗값 기반 이진화 Mock
def binarize_image_by_threshold(data: np.ndarray, threshold: float = 0.5) -> np.ndarray:
    return (data > threshold).astype(np.float32)

# AI 모델 Mock (출력 쉐이프 검증용)
class MockModel:
    def __init__(self, output_shape): self.output_shape = output_shape
    def predict(self, input_array, verbose=0): return np.zeros(self.output_shape)

def unet3d(input_shape: tuple):
    # Segmentation: 입력과 동일한 쉐이프 출력
    return MockModel(output_shape=(1, *input_shape[:-1], 1))

def cnn3d(input_shape: tuple):
    # Classification: (Batch_size, 1) 쉐이프 출력
    return MockModel(output_shape=(1, 1))

# --------------------------------------------------------------------------------
# ICV 보정 부피 로직
# --------------------------------------------------------------------------------
def calculate_icv_normalized_volume(raw_volume: float, patient_icv: float, mean_icv: float = 1500000.0) -> float:
    """ICV 보정 해마 부피 계산 로직 (V_norm = V_raw * (Mean ICV / Patient ICV))"""
    if patient_icv <= 0: return 0.0
    return raw_volume * (mean_icv / patient_icv)


In [ ]:


# ================================================================================
# 5.1 MRI 파일 업로드 및 환자 정보 기능
# ================================================================================
def test_d_01_dicom_format_validation():
# 데이터가 표준 DICOM 포맷(.dcm)으로 수집되었는지 확인
    assert upload_mri_request('mri_data_001.dcm', {'id': 'P001'}) is True
    assert upload_mri_request('data.nii', {'id': 'P001'}) is False


# ================================================================================
# 5.2 데이터 전처리 및 정규화 기능
# ================================================================================
def test_nor_01_z_normalization():
# Z-score 정규화가 올바르게 수행되는지 확인
    arr = np.array([1, 2, 3, 4, 5], dtype=np.float32)
    norm = z(arr)
    assert np.isclose(np.mean(norm), 0, atol=1e-5)
    assert np.isclose(np.std(norm), 1, atol=1e-5)

def test_nor_02_crop_or_pad_shape():
# 3D 데이터의 Numpy 배열 크기가 정규화되어 저장되는지 확인
    # 패딩 테스트
    small_arr = np.ones((50, 50, 50), dtype=np.float32)
    out_pad = crop_or_pad_to((100, 100, 100), small_arr)
    assert out_pad.shape == (100, 100, 100)

    # 크롭 테스트
    large_arr = np.ones((150, 150, 150), dtype=np.float32)
    out_crop = crop_or_pad_to((100, 100, 100), large_arr)
    assert out_crop.shape == (100, 100, 100)

def test_nor_03_background_removal():
# 데이터에서 불필요한 구조(배경 등)가 제거되었는지 확인
    data = np.array([0.01, 0.5, 0.9, 0.05], dtype=np.float32)
    cleaned_data = remove_skull_and_background(data, threshold=0.1)
    expected = np.array([0.0, 0.5, 0.9, 0.0], dtype=np.float32)
    assert np.array_equal(cleaned_data, expected), "배경 제거 로직 오류"

# --------------------------------------------------------------------------------
# +) 데이터 분할 비율 검증 (70:10:20 요구사항)
# --------------------------------------------------------------------------------
def test_data_split_ratio_70_10_20():
# 학습 70%, 검증 10%, 테스트 20% 비율로 데이터 분할이 정상적으로 이루어지는지 검증
    total_samples = 1000
    X = np.arange(total_samples)
    X_train_val, X_test = train_test_split(X, test_size=0.2, random_state=42)
    X_train, X_val = train_test_split(X_train_val, test_size=0.125, random_state=42)

    assert len(X_train) == pytest.approx(700, abs=1)
    assert len(X_val) == pytest.approx(100, abs=1)
    assert len(X_test) == pytest.approx(200, abs=1)


# ================================================================================
# 5.3 해마 자동 분할 및 세그멘테이션
# ================================================================================
def test_seg_01_binarization_logic():
# 임곗값을 기준으로 해마 영역을 이진화하여 추출할 수 있는지 확인
    data = np.array([0.1, 0.6, 0.4, 0.9], dtype=np.float32)
    binarized = binarize_image_by_threshold(data, threshold=0.5)
    expected = np.array([0.0, 1.0, 0.0, 1.0], dtype=np.float32)
    assert np.array_equal(binarized, expected), "임곗값 기반 이진화 로직 오류"

def test_seg_02_unet_output_shape():
# 모델이 해마 영역을 자동으로 세그멘테이션할 수 있는지 (U-Net 구조 확인)
    model = unet3d(input_shape=(128, 128, 128, 1))
    out = model.predict(np.zeros((1, 128, 128, 128, 1)))
    assert out.shape == (1, 128, 128, 128, 1), "U-Net 모델 출력 쉐이프 불일치"

def test_seg_03_volume_split_logic():
# AI 기반으로 좌/우 부피를 자동 분할하고 측정하는 기능이 정상 작동하는지 확인
    seg_map = np.ones((10, 10, 10))
    volumes = calculate_volume_and_split(seg_map)
    assert 'left_vol' in volumes and 'right_vol' in volumes, "좌/우 부피 분할 측정 로직 실패"


# ================================================================================
# 5.4 정량 피처 추출 기능
# ================================================================================
def test_nfe_02_asymmetry_accuracy():
# 추출된 비대칭 지수 등의 정량 피처가 정확한지 확인
    # Case 1: 좌우 부피가 같을 때
    assert extract_asymmetry_index(500, 500) == pytest.approx(0.0)
    # Case 2: 좌우 부피가 다를 때
    assert extract_asymmetry_index(100, 500) == pytest.approx(400.0 / 300.0)

def test_icv_normalized_volume_calculation():
# ICV 보정 해마 부피 계산 로직이 정확한지 확인
    raw_vol = 3000.0
    mean_icv = 1500000.0

    # 환자 ICV가 평균과 같을 때
    norm_vol_1 = calculate_icv_normalized_volume(raw_vol, mean_icv)
    assert norm_vol_1 == pytest.approx(raw_vol)

    # 환자 ICV가 평균의 1/2일 때
    patient_icv_2 = 750000.0
    norm_vol_2 = calculate_icv_normalized_volume(raw_vol, patient_icv_2)
    assert norm_vol_2 == pytest.approx(raw_vol * 2.0)

def test_nfe_03_feature_transfer():
# 추출된 피처가 분류 모델 학습의 입력으로 정상 전달되는지 확인
    features = {'volume': 100, 'index': 20}
    assert check_feature_transfer(features) is True, "분류 모델로의 피처 전달 실패"


# ================================================================================
# 5.5 3D 해마 모델링 및 입체적 확인 기능
# ================================================================================
@pytest.mark.performance
def test_vis_01_3d_viewer_speed():
# 3D 모델링 가능하게 해야 함 (성능)
    data = np.random.randint(0, 2, (64, 64, 64))
    TARGET_TIME_SEC = 5.0
    start_time = time.time()
    generate_3d_viewer_performance(data)
    elapsed_time = time.time() - start_time
    assert elapsed_time < TARGET_TIME_SEC, "3D 뷰어 렌더링 속도 미달"


# ================================================================================
# 5.6 분류 예측 모델 로드 및 환자 상태 예측
# ================================================================================
def test_pre_01_pre_02_cnn3d_output_shape():
# 3D AI 모델을 통한 예측 가능 (CNN 구조 확인 및 확률 예측 기능)
    model = cnn3d(input_shape=(96, 96, 96, 1))
    out = model.predict(np.zeros((1, 96, 96, 96, 1)))
    assert out.shape == (1, 1), "CNN 분류 모델 출력 쉐이프 불일치"

@pytest.mark.performance
def test_pre_03_segmentation_speed():
# 실시간 또는 준실시간 처리가 가능한지 확인 (성능)
    data = np.random.rand(128, 128, 128)
    TARGET_TIME_SEC = 180.0
    start_time = time.time()
    segment_hippocampus_performance(data)
    elapsed_time = time.time() - start_time

    assert elapsed_time <= TARGET_TIME_SEC + 1.0, f"세그멘테이션 속도 미달: {elapsed_time:.2f}초 > {TARGET_TIME_SEC:.1f}초"


# ================================================================================
# 5.7 결과 보고서 시각화 및 데이터 저장
# ================================================================================
def test_rvd_01_prediction_format():
# AI 예측 결과(분류 확률)가 정확하게 표시되는지 확인 (포맷팅)]
    prob_result = {'AD': 0.75, 'CN': 0.25}
    formatted_str = format_prediction_for_ui(prob_result)
    assert "AD: 75.0%" in formatted_str, "예측 결과 포맷팅 로직 오류"

def test_rvd_03_save_data_validation():
# 최종 결과가 정상적으로 저장되고 관리되는지 확인
    assert save_final_result({"id": "P001"}, {"auc": 0.9}) is True
    assert save_final_result({"id": "P001"}, None) is False